# E47 — A explicação que a divergência escolhe

O capítulo contou quantas explicações cabem e mediu a folga de cada uma; a região não escolhe.
A travessia pergunta à lei inteira dos blocos: projetada sobre a envoltória convexa das
180 leis da grade persistente, a projeção em divergência --- que a convexidade garante existir
e ser única --- escolhe o mesmo ponto que a tolerância?

Cada uma das réplicas re-sorteia a grade (a semente da família anda com a réplica; o dado é o
sp500.csv fixo), refaz as folgas --- a escolha da tolerância é por réplica --- e projeta.

In [1]:
import sys
sys.path.insert(0, "lib")

import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from frevolab import dados, evidencia, regimes, volatilidade, graficos

SEMENTE = 114          # <- brinque com:
REPLICAS = 12          # <- brinque com: quantas grades independentes
ALFABETO = 25          # <- brinque com: as casas da contagem; a última engloba a cauda
CONTROLE = 20          # o segundo agrupamento, para a sensibilidade declarada

In [2]:
# O dado fixo, sem sorte: a série do livro, as quatro estatísticas e a lei inteira dos blocos.
retornos = volatilidade.retornos_log(dados.carregar_serie(evidencia.SERIE_PADRAO))
x = retornos.to_numpy()
sigma = float(x.std(ddof=1))
real = regimes.estatisticas(x)
lei_dado = regimes.lei_dos_blocos(x, alfabeto=ALFABETO)
casas_dado = np.nonzero(lei_dado)[0]
print("dado: taxa=%.4f pior=%d mediana=%.1f acima=%.4f | casas com massa: %d a %d"
      % (real["taxa"], real["pior"], real["mediana"], real["acima_do_dobro"],
         casas_dado.min(), casas_dado.max()))

# as quatro estatísticas como funcionais da lei: a taxa é a média da contagem pelo bloco, o
# pior é a casa mais alta com massa, a mediana é a casa onde a acumulada cruza metade e o
# excesso é a massa das casas acima do dobro da promessa.
funcional = lambda lei: {
    "taxa": float((lei * np.arange(lei.size)).sum()) / regimes.BLOCO_PADRAO,
    "pior": float(np.max(np.nonzero(lei)[0])),
    "mediana": float(np.arange(lei.size)[np.searchsorted(np.cumsum(lei), 0.5)]),
    "acima_do_dobro": float(lei[int(2 * regimes.BLOCO_PADRAO * regimes.TAXA_PADRAO) + 1:].sum())}
lei_dado_c = regimes.lei_dos_blocos(x, alfabeto=CONTROLE)
print("funcionais da lei do dado:", {k: round(v, 4) for k, v in funcional(lei_dado).items()})

dado: taxa=0.0513 pior=20 mediana=2.0 acima=0.1305 | casas com massa: 0 a 20
funcionais da lei do dado: {'taxa': 0.0515, 'pior': 20.0, 'mediana': 2.0, 'acima_do_dobro': 0.1305}


In [3]:
# As réplicas: cada uma re-sorteia a grade de 180 membros, estima as leis nos dois alfabetos,
# refaz as folgas (a escolha da tolerância é por réplica) e projeta no alfabeto principal.
TOL, CHAVES = evidencia.TOLERANCIA_PADRAO, evidencia.CHAVES_PADRAO
linhas = []
for i in range(REPLICAS):
    sorteio = np.random.default_rng(SEMENTE + 100 + i)
    folgas, leis, leis_c, params = [], [], [], []
    for p_ in evidencia.GRADE_P_PADRAO:
        for razao in evidencia.GRADE_RAZAO_PADRAO:
            for perm in evidencia.GRADE_PERMANENCIA_PADRAO:
                s = regimes.persistente(len(x), sorteio, sigma, p_, razao, perm)
                e = regimes.estatisticas(s)
                folgas.append(max(abs(e[k] - real[k]) / TOL[k] for k in CHAVES))
                leis.append(regimes.lei_dos_blocos(s, alfabeto=ALFABETO))
                leis_c.append(regimes.lei_dos_blocos(s, alfabeto=CONTROLE))
                params.append((p_, razao, perm))
    folgas, leis, leis_c = np.array(folgas), np.array(leis), np.array(leis_c)
    esc = int(np.argmin(folgas))
    proj = regimes.projecao(lei_dado, leis)
    proj_c = regimes.projecao(lei_dado_c, leis_c)
    fp, fe = funcional(proj["lei"]), funcional(leis[esc])
    sep = {k: abs(fp[k] - fe[k]) / TOL[k] for k in CHAVES}
    pesos_params = np.array([list(q) for q in params])
    linhas.append({
        "cabem": int((folgas <= 1.0).sum()), "escolha": esc,
        "escolha_params": params[esc], "escolha_folga": float(folgas[esc]),
        "div_projetada": proj["divergencia"], "div_controle": proj_c["divergencia"],
        "dist_nats": regimes.divergencia(leis[esc], proj["lei"]),
        "casas_zero": int(((lei_dado > 0) & (leis[esc] <= 0)).sum()),
        "peso_dominante": proj["peso_dominante"], "peso_controle": proj_c["peso_dominante"],
        "vertice": params[proj["vertice_dominante"]],
        "baricentro": tuple(float((proj["pesos"] * pesos_params[:, j]).sum()) for j in range(3)),
        "baricentro_c": tuple(float((proj_c["pesos"] * pesos_params[:, j]).sum()) for j in range(3)),
        "separacao": max(sep.values()),
        "separacao_pior": sep["pior"],
        "separacao_mediana": sep["mediana"],
        "casas_sep": [int(k) for k in range(ALFABETO)
                      if abs(leis[esc][k] - lei_dado[k]) > 0.004
                      and abs(proj["lei"][k] - lei_dado[k]) < 0.5 * abs(leis[esc][k] - lei_dado[k])]})
    if i == 0:
        leis0, params0, folgas0, proj0, esc0 = leis, params, folgas, proj, esc
    L = linhas[-1]
    print("rep %2d: cabem=%d escolha=%s folga=%.3f | div=%.4f nats, controle %.4f | "
          "dist=%.4f | casas zero=%d | peso=%.3f, controle %.3f | vertice=%s | bar=(%.3f, %.2f, %.0f)"
          % (i + 1, L["cabem"], L["escolha_params"], L["escolha_folga"], L["div_projetada"],
             L["div_controle"], L["dist_nats"], L["casas_zero"], L["peso_dominante"],
             L["peso_controle"], L["vertice"], L["baricentro"][0], L["baricentro"][1],
             L["baricentro"][2]))

med = lambda k: float(np.median([l[k] for l in linhas]))
vert_mais = max(set(l["vertice"] for l in linhas), key=[l["vertice"] for l in linhas].count)
estavel = sum(1 for l in linhas if l["vertice"] == vert_mais)
print()
print("RESUMO das %d réplicas:" % REPLICAS)
print("  tolerância admite: mediana %.1f, máximo %d" % (med("cabem"), max(l["cabem"] for l in linhas)))
print("  divergência da projeção: %.4f nats por bloco (controle: %.4f)"
      % (med("div_projetada"), med("div_controle")))
print("  distância da escolha à projeção: %.4f nats | casas zero da escolha: mediana %.1f"
      % (med("dist_nats"), med("casas_zero")))
print("  separação nas estatísticas: %.2f tolerâncias (máximo das quatro)"
      % med("separacao"))
print("  peso dominante: mediana %.3f, máximo %.3f | vértice modal %s domina %d de %d réplicas"
      % (med("peso_dominante"), max(l["peso_dominante"] for l in linhas), vert_mais, estavel, REPLICAS))
print("  baricentro: p=%.3f, razão=%.2f, permanência=%.0f (controle: %.3f, %.2f)"
      % (float(np.median([l["baricentro"][0] for l in linhas])),
         float(np.median([l["baricentro"][1] for l in linhas])),
         float(np.median([l["baricentro"][2] for l in linhas])),
         float(np.median([l["baricentro_c"][0] for l in linhas])),
         float(np.median([l["baricentro_c"][1] for l in linhas]))))

rep  1: cabem=1 escolha=(0.18, 2.5, 120.0) folga=1.000 | div=0.0130 nats, controle 0.0129 | dist=0.0198 | casas zero=2 | peso=0.383, controle 0.377 | vertice=(0.18, 3.0, 30.0) | bar=(0.165, 3.24, 32)


rep  2: cabem=0 escolha=(0.18, 3.0, 60.0) folga=1.005 | div=0.0119 nats, controle 0.0109 | dist=0.0291 | casas zero=1 | peso=0.274, controle 0.239 | vertice=(0.25, 3.0, 60.0) | bar=(0.181, 3.09, 41)


rep  3: cabem=3 escolha=(0.12, 4.0, 60.0) folga=0.968 | div=0.0099 nats, controle 0.0095 | dist=0.0379 | casas zero=0 | peso=0.319, controle 0.314 | vertice=(0.25, 3.5, 30.0) | bar=(0.171, 2.91, 51)


rep  4: cabem=4 escolha=(0.25, 4.0, 60.0) folga=0.744 | div=0.0066 nats, controle 0.0056 | dist=0.0320 | casas zero=1 | peso=0.303, controle 0.318 | vertice=(0.08, 4.0, 30.0) | bar=(0.136, 3.37, 39)


rep  5: cabem=2 escolha=(0.12, 4.0, 60.0) folga=0.500 | div=0.0081 nats, controle 0.0075 | dist=0.0363 | casas zero=0 | peso=0.394, controle 0.401 | vertice=(0.12, 4.0, 30.0) | bar=(0.164, 3.66, 47)


...

rep  8: cabem=4 escolha=(0.12, 4.0, 10.0) folga=0.619 | div=0.0084 nats, controle 0.0079 | dist=0.0209 | casas zero=1 | peso=0.294, controle 0.292 | vertice=(0.18, 4.0, 30.0) | bar=(0.171, 3.57, 43)


rep  9: cabem=3 escolha=(0.08, 3.5, 30.0) folga=0.874 | div=0.0090 nats, controle 0.0082 | dist=0.0294 | casas zero=1 | peso=0.353, controle 0.362 | vertice=(0.25, 2.5, 30.0) | bar=(0.207, 3.19, 42)


rep 10: cabem=0 escolha=(0.05, 3.0, 30.0) folga=1.212 | div=0.0131 nats, controle 0.0127 | dist=0.0801 | casas zero=2 | peso=0.459, controle 0.474 | vertice=(0.08, 4.0, 10.0) | bar=(0.138, 3.86, 31)


rep 11: cabem=2 escolha=(0.08, 4.0, 30.0) folga=0.713 | div=0.0110 nats, controle 0.0099 | dist=0.0311 | casas zero=0 | peso=0.328, controle 0.340 | vertice=(0.12, 3.5, 30.0) | bar=(0.146, 3.45, 57)


rep 12: cabem=1 escolha=(0.12, 4.0, 10.0) folga=1.000 | div=0.0127 nats, controle 0.0124 | dist=0.0475 | casas zero=0 | peso=0.244, controle 0.255 | vertice=(0.12, 3.0, 30.0) | bar=(0.161, 3.34, 53)

RESUMO das 12 réplicas:
  tolerância admite: mediana 2.5, máximo 4
  divergência da projeção: 0.0095 nats por bloco (controle: 0.0088)
  distância da escolha à projeção: 0.0315 nats | casas zero da escolha: mediana 1.0
  separação nas estatísticas: 2.00 tolerâncias (máximo das quatro)
  peso dominante: mediana 0.340, máximo 0.519 | vértice modal (0.08, 4.0, 30.0) domina 2 de 12 réplicas
  baricentro: p=0.164, razão=3.35, permanência=41 (controle: 0.164, 3.36)


In [4]:
# Figura 1: o vale da divergência no plano p × razão, na permanência do vértice dominante
# da primeira réplica. A estrela é o baricentro da projeção; o círculo aberto é a escolha da
# tolerância --- pontos diferentes, e a escolha vive entre os marcadores abertos: a divergência
# do dado à sua lei é infinita, porque o dado visita casas onde ela não põe massa.
perm_vencedora = params0[proj0["vertice_dominante"]][2]
fatiados = [j for j in range(len(params0)) if params0[j][2] == perm_vencedora]
divs = np.array([regimes.divergencia(lei_dado, leis0[j]) for j in fatiados])
ps = np.array([params0[j][0] for j in fatiados])
rs = np.array([params0[j][1] for j in fatiados])
finitos = np.isfinite(divs)
bar_p = float(sum(w * params0[j][0] for j, w in enumerate(proj0["pesos"])))
bar_r = float(sum(w * params0[j][1] for j, w in enumerate(proj0["pesos"])))
fig, eixo = plt.subplots(figsize=(7.0, 4.6))
eixo.scatter(ps[finitos], rs[finitos], s=90, c=np.log10(divs[finitos]), cmap="viridis_r",
             edgecolor="#222222", zorder=4, label="divergência finita (o vale)")
eixo.scatter(ps[~finitos], rs[~finitos], s=55, marker="o", facecolor="none",
             edgecolor="#999999", zorder=3, label="divergência infinita: casa que o dado visita sem massa")
eixo.scatter([bar_p], [bar_r], marker="*", s=420, color="#b03a2e", edgecolor="#222222",
             zorder=6, label="o baricentro da projeção")
eixo.scatter([params0[esc0][0]], [params0[esc0][1]], s=150, marker="o", facecolor="none",
             edgecolor="#b03a2e", linewidths=1.8, zorder=6,
             label="a escolha da tolerância (ponto distinto, na permanência %s)"
                   % ("%.0f" % params0[esc0][2] if params0[esc0][2] >= 10 else "%.1f" % params0[esc0][2]))
eixo.set_xlabel("p: a fração de dias no regime agitado")
eixo.set_ylabel("a razão entre as duas leis")
eixo.set_title("permanência %.0f: a fatia do vértice dominante da primeira réplica" % perm_vencedora,
               fontsize=9)
eixo.legend(frameon=False, fontsize=7.5, loc="upper left")
eixo.grid(alpha=0.2)
fig.tight_layout()
graficos.salvar(fig, "E47_projecao", 1)
plt.show()

/tmp/ipykernel_218124/1925419235.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Figura 2: as três leis sobre o alfabeto da contagem --- o dado, a projeção e a escolha da
# tolerância. A escala é logarítmica porque a massa cai por décadas; a curva da tolerância
# acaba antes do fim do alfabeto (as casas que o dado visita sem massa dela), e a da projeção
# acompanha o dado justamente nas casas onde a da tolerância se separa.
casas = np.arange(ALFABETO)
separam = sorted(set(c for l in linhas for c in l["casas_sep"]))
bandas = linhas[0]["casas_sep"]
fig, eixo = plt.subplots(figsize=(7.6, 4.4))
eixo.plot(casas, lei_dado, "o-", lw=1.6, color="#1f4e79", label="a lei do dado", ms=4)
eixo.plot(casas, proj0["lei"], "s--", lw=1.5, color="#2e7d32", label="a lei projetada", ms=4)
eixo.plot(casas, leis0[esc0], "^:", lw=1.5, color="#b03a2e", label="a lei da escolha da tolerância", ms=4)
for casa in bandas:
    eixo.axvline(casa, color="#b03a2e", alpha=0.15, lw=6)
eixo.set_yscale("log")
eixo.set_xlabel("casas do alfabeto: rompimentos no bloco (a última engloba a cauda)")
eixo.set_ylabel("a massa de cada casa")
eixo.set_ylim(1e-6, 1.0)
eixo.legend(frameon=False, fontsize=8.5)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E47_projecao", 2)
plt.show()
print("casas onde a tolerância se separa e a projeção acompanha (uniao das réplicas): %s" % separam)
print("faixas marcadas nesta figura (primeira réplica): %s" % bandas)
print("a lei da escolha da tolerância e zero nas casas: %s"
      % [int(k) for k in range(ALFABETO) if leis0[esc0][k] == 0 and lei_dado[k] > 0])

casas onde a tolerância se separa e a projeção acompanha (uniao das réplicas): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
faixas marcadas nesta figura (primeira réplica): [0, 3, 4, 6]
a lei da escolha da tolerância e zero nas casas: [19, 20]


/tmp/ipykernel_218124/1871805147.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Leitura visual das figuras

Leitura a fazer contra o PNG depois de executar o caderno (AGENTS.md §9); sessão sem entrada
de imagem declara a falta e não inventa a leitura. O que conferir:

1. **Figura 1** --- a estrela (baricentro da projeção) e o círculo aberto (escolha da
   tolerância) estão em pontos distintos do plano p × razão; o círculo está entre os
   marcadores abertos de divergência infinita, e a estrela, na região de divergência finita.
2. **Figura 2** --- as três curvas descem por décadas (escala logarítmica); a curva da
   escolha da tolerância termina antes do fim do alfabeto, cortada nas casas onde o dado põe
   massa e ela não põe; a curva projetada segue o dado nas casas marcadas, onde a da
   tolerância se separa.

In [6]:
# O que sai do laboratório e o que o livro cita. A direção da distância em nats é declarada:
# da lei da escolha da tolerância para a lei projetada --- a direção contrária é infinita
# (as casas zero), e o número de casas zero é o que o livro imprime dessa infinitude.
resultado = {
    "projecao_replicas": int(REPLICAS),
    "projecao_tolerancia_cabem": round(med("cabem"), 1),
    "projecao_divergencia_projetada": round(med("div_projetada"), 4),
    "projecao_distancia_nats": round(med("dist_nats"), 4),
    "projecao_escolha_casas_zero": round(med("casas_zero"), 1),
    "projecao_separacao_tolerancias": round(med("separacao"), 2),
    "projecao_separacao_pior": round(med("separacao_pior"), 2),
    "projecao_separacao_mediana": round(med("separacao_mediana"), 2),
    "projecao_escolha_fora": int(sum(1 for l in linhas if l["casas_zero"] > 0)),
    "projecao_peso_dominante": round(med("peso_dominante"), 3),
    "projecao_peso_dominante_max": round(max(l["peso_dominante"] for l in linhas), 3),
    "projecao_vertice_estavel": int(estavel),
    "projecao_baricentro_p": round(float(np.median([l["baricentro"][0] for l in linhas])), 3),
    "projecao_baricentro_razao": round(float(np.median([l["baricentro"][1] for l in linhas])), 2),
    "projecao_baricentro_permanencia": round(float(np.median([l["baricentro"][2] for l in linhas])), 1),
    "projecao_controle_divergencia": round(med("div_controle"), 4),
    "projecao_controle_baricentro_p": round(float(np.median([l["baricentro_c"][0] for l in linhas])), 3),
}
caminho = Path("lab/resultados/E47_projecao.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1, sort_keys=True))
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

{
 "projecao_baricentro_p": 0.164,
 "projecao_baricentro_permanencia": 41.3,
 "projecao_baricentro_razao": 3.35,
 "projecao_controle_baricentro_p": 0.164,
 "projecao_controle_divergencia": 0.0088,
 "projecao_distancia_nats": 0.0315,
 "projecao_divergencia_projetada": 0.0095,
 "projecao_escolha_casas_zero": 1.0,
 "projecao_escolha_fora": 7,
 "projecao_peso_dominante": 0.34,
 "projecao_peso_dominante_max": 0.519,
 "projecao_replicas": 12,
 "projecao_separacao_mediana": 0.0,
 "projecao_separacao_pior": 2.0,
 "projecao_separacao_tolerancias": 2.0,
 "projecao_tolerancia_cabem": 2.5,
 "projecao_vertice_estavel": 2
}
lab/resultados/E47_projecao.json gravado | 17 grandezas
